In [1]:
import pandas as pd
import os as os
# Data Cleaning for Recipe Dataset
import pandas as pd
import numpy as np
import re
import os
import kagglehub as kagglehub

from difflib import SequenceMatcher
from collections import Counter

import ast

/Users/makabaka/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#res 

dataset_dir = "/Users/makabaka/.cache/kagglehub/datasets/elisaxxygao/foodrecsysv1/versions/1"

raw_recipes = pd.read_csv(os.path.join(dataset_dir, "raw-data_recipe.csv"))
core_recipes = pd.read_csv(os.path.join(dataset_dir, "core-data_recipe.csv"))

raw_interactions = pd.read_csv(os.path.join(dataset_dir, "raw-data_interaction.csv"))

train_df = pd.read_csv(os.path.join(dataset_dir, "core-data-train_rating.csv"))
valid_df = pd.read_csv(os.path.join(dataset_dir, "core-data-valid_rating.csv"))
test_df  = pd.read_csv(os.path.join(dataset_dir, "core-data-test_rating.csv"))


#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)
#pd.set_option('display.max_colwidth', None)


print("=== Recipe Data ===")
print("Raw recipes:", raw_recipes.shape)
print("Core recipes:", core_recipes.shape)

print("\n=== Interaction Data ===")
print("Raw interactions:", raw_interactions.shape)
print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

print("\n=== Columns Preview ===")
print("Raw recipes columns:", raw_recipes.columns.tolist())
print("Interactions columns:", raw_interactions.columns.tolist())

print("\n=== Sample Raw Recipes ===")
display(raw_recipes.head())

print("\n=== Sample Train Interactions ===")
display(train_df.head())

=== Recipe Data ===
Raw recipes: (49698, 9)
Core recipes: (45630, 6)

=== Interaction Data ===
Raw interactions: (3794003, 4)
Train: (676946, 4)
Valid: (133459, 4)
Test : (283440, 4)

=== Columns Preview ===
Raw recipes columns: ['recipe_id', 'recipe_name', 'aver_rate', 'image_url', 'review_nums', 'ingredients', 'cooking_directions', 'nutritions', 'reviews']
Interactions columns: ['user_id', 'recipe_id', 'rating', 'dateLastModified']

=== Sample Raw Recipes ===


,recipe_id,recipe_name,aver_rate,image_url,review_nums,ingredients,cooking_directions,nutritions,reviews
0,222388,Homemade Bacon,5.000000,https://images.media-allrecipes.com/userphotos...,3,pork belly^smoked paprika^kosher salt^ground b...,{'directions': u'Prep\n5 m\nCook\n2 h 45 m\nRe...,"{u'niacin': {u'hasCompleteData': False, u'name...","{8542392: {'rating': 5, 'followersCount': 11, ..."
1,240488,"Pork Loin, Apples, and Sauerkraut",4.764706,https://images.media-allrecipes.com/userphotos...,29,sauerkraut drained^Granny Smith apples sliced^...,{'directions': u'Prep\n15 m\nCook\n2 h 30 m\nR...,"{u'niacin': {u'hasCompleteData': False, u'name...","{3574785: {'rating': 5, 'followersCount': 0, '..."
2,218939,Foolproof Rosemary Chicken Wings,4.571429,https://images.media-allrecipes.com/userphotos...,12,chicken wings^sprigs rosemary^head garlic^oliv...,"{'directions': u""Prep\n20 m\nCook\n40 m\nReady...","{u'niacin': {u'hasCompleteData': True, u'name'...","{13774946: {'rating': 5, 'followersCount': 0, ..."
3,87211,Chicken Pesto Paninis,4.625000,https://images.media-allrecipes.com/userphotos...,163,focaccia bread quartered^prepared basil pesto^...,{'directions': u'Prep\n15 m\nCook\n5 m\nReady ...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{1563136: {'rating': 5, 'followersCount': 0, '..."
4,245714,Potato Bacon Pizza,4.500000,https://images.media-allrecipes.com/userphotos...,2,red potatoes^strips bacon^Sauce:^heavy whippin...,{'directions': u'Prep\n20 m\nCook\n45 m\nReady...,"{u'niacin': {u'hasCompleteData': True, u'name'...","{2945555: {'rating': 5, 'followersCount': 6690..."



=== Sample Train Interactions ===


,user_id,recipe_id,rating,dateLastModified
0,5215572,17991,5,2010-08-25T14:38:53.84\n
1,5215572,170724,4,2010-09-09T14:04:45.733\n
2,5215572,18045,5,2010-08-16T14:51:25.833\n
3,3622615,60598,4,2009-03-15T12:10:20.85\n
4,1313770,47519,5,2005-10-04T15:43:36.653\n


In [3]:
comment_column = [
    "recipe_id",
    "recipe_name",
    "review_nums",
    "reviews",
    "aver_rate"
]

comment_1 =raw_recipes.copy()


comment_1 =comment_1[comment_column]
comment_1["reviews_dict"] = comment_1["reviews"].apply(lambda x: ast.literal_eval(x) if pd.notna(x) else {})


In [4]:
def expand_reviews(row):
    reviews = row["reviews_dict"]
    rows = []
    for user_id, review_data in reviews.items():
        temp = row.to_dict()

        # add review info
        temp["user_id"] = user_id
        temp.update(review_data)

        rows.append(temp)

    return rows

In [5]:
expanded = comment_1.apply(expand_reviews, axis=1)

# flatten list of lists
expanded = [item for sublist in expanded for item in sublist]

comment_2 = pd.DataFrame(expanded)

In [6]:
comment_2.head()

,recipe_id,recipe_name,review_nums,reviews,aver_rate,reviews_dict,user_id,rating,followersCount,madeRecipesCount,favoritesCount,dateLastModified,text,followingCount
0,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",8542392,5,11,18,200,2017-04-22T12:46:43.663,Best breakfast ever! I ran out of paprika whil...,0
1,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",11174581,5,8,55,101,2013-06-20T15:50:25.96,Awesome!\nIt's amazing.,0
2,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",8262477,5,0,1,52,2015-02-14T07:27:51.307,The flavors came together well and it really w...,0
3,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,"{3574785: {'rating': 5, 'followersCount': 0, '...",3574785,5,0,4,118,2017-10-07T18:20:08.973,"Like most, I changed it a bit. Not a fan of T...",0
4,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,"{3574785: {'rating': 5, 'followersCount': 0, '...",12145410,2,0,83,170,2018-01-06T00:06:09.563,This was one of the worst recipes I have ever ...,0


In [7]:
comment_2['user_id'].value_counts()

user_id
2043209    4003
1153011    2496
2945555    2148
268713     2045
827351     1800
           ... 
3916806       1
5387840       1
7523907       1
1750110       1
5912708       1
Name: count, Length: 1160267, dtype: int64

In [8]:
comment_2['source']="foodrecsys"
comment_2.head()

,recipe_id,recipe_name,review_nums,reviews,aver_rate,reviews_dict,user_id,rating,followersCount,madeRecipesCount,favoritesCount,dateLastModified,text,followingCount,source
0,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",8542392,5,11,18,200,2017-04-22T12:46:43.663,Best breakfast ever! I ran out of paprika whil...,0,foodrecsys
1,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",11174581,5,8,55,101,2013-06-20T15:50:25.96,Awesome!\nIt's amazing.,0,foodrecsys
2,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,"{8542392: {'rating': 5, 'followersCount': 11, ...",8262477,5,0,1,52,2015-02-14T07:27:51.307,The flavors came together well and it really w...,0,foodrecsys
3,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,"{3574785: {'rating': 5, 'followersCount': 0, '...",3574785,5,0,4,118,2017-10-07T18:20:08.973,"Like most, I changed it a bit. Not a fan of T...",0,foodrecsys
4,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,"{3574785: {'rating': 5, 'followersCount': 0, '...",12145410,2,0,83,170,2018-01-06T00:06:09.563,This was one of the worst recipes I have ever ...,0,foodrecsys


In [9]:
comment_2 = comment_2.drop(columns=['reviews_dict'], errors='ignore')

In [10]:
comment_2.head()

,recipe_id,recipe_name,review_nums,reviews,aver_rate,user_id,rating,followersCount,madeRecipesCount,favoritesCount,dateLastModified,text,followingCount,source
0,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,8542392,5,11,18,200,2017-04-22T12:46:43.663,Best breakfast ever! I ran out of paprika whil...,0,foodrecsys
1,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,11174581,5,8,55,101,2013-06-20T15:50:25.96,Awesome!\nIt's amazing.,0,foodrecsys
2,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.000000,8262477,5,0,1,52,2015-02-14T07:27:51.307,The flavors came together well and it really w...,0,foodrecsys
3,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,3574785,5,0,4,118,2017-10-07T18:20:08.973,"Like most, I changed it a bit. Not a fan of T...",0,foodrecsys
4,240488,"Pork Loin, Apples, and Sauerkraut",29,"{3574785: {'rating': 5, 'followersCount': 0, '...",4.764706,12145410,2,0,83,170,2018-01-06T00:06:09.563,This was one of the worst recipes I have ever ...,0,foodrecsys


In [11]:
comment_2.shape

(3794003, 14)

In [12]:
#review dataset summary
def dataset_summary(df):
    
    def safe_nunique(col):
        try:
            return col.nunique()
        except TypeError:
            return col.astype(str).nunique()
    
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isnull().sum(),
        "missing_pct": (df.isnull().sum() / len(df) * 100).round(2),
        "n_unique": df.apply(safe_nunique)
    }).sort_values(by="missing_count", ascending=False)

    return summary

In [13]:
#dataset_summary(comment_2)

In [14]:
comment_2.drop(columns=["followingCount"], inplace=True)
comment_2.rename(columns={"text": "comment"}, inplace=True)

In [15]:
comment_2['dateLastModified'] = pd.to_datetime(
    comment_2['dateLastModified'],
    format='mixed',
    errors='coerce'
)

In [16]:
comment_2.head(1)

,recipe_id,recipe_name,review_nums,reviews,aver_rate,user_id,rating,followersCount,madeRecipesCount,favoritesCount,dateLastModified,comment,source
0,222388,Homemade Bacon,3,"{8542392: {'rating': 5, 'followersCount': 11, ...",5.0,8542392,5,11,18,200,2017-04-22 12:46:43.663,Best breakfast ever! I ran out of paprika whil...,foodrecsys


In [17]:
# Download dataset files to a local folder
path = kagglehub.dataset_download("irkaal/foodcom-recipes-and-reviews")

print("Dataset downloaded to:", path)
print("Files:", os.listdir(path))

# Load CSVs
reviews = pd.read_csv(os.path.join(path, "reviews.csv"))

recipes = pd.read_csv(os.path.join(path, "recipes.csv"))

reviews.head()
#print(reviews.shape)

Dataset downloaded to: /Users/makabaka/.cache/kagglehub/datasets/irkaal/foodcom-recipes-and-reviews/versions/2
Files: ['reviews.csv', 'recipes.csv', 'recipes.parquet', 'reviews.parquet']


,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25T21:44:00Z,2000-01-25T21:44:00Z
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17T16:49:59Z,2001-10-17T16:49:59Z
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25T09:00:00Z,2000-02-25T09:00:00Z
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13T21:15:00Z,2000-03-13T21:15:00Z
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28T12:51:00Z,2000-03-28T12:51:00Z


In [27]:
reviews.shape

(1401982, 9)

In [18]:
reviews["source"] = "foodcom"

In [19]:
recipes.columns.tolist()

['RecipeId',
 'Name',
 'AuthorId',
 'AuthorName',
 'CookTime',
 'PrepTime',
 'TotalTime',
 'DatePublished',
 'Description',
 'Images',
 'RecipeCategory',
 'Keywords',
 'RecipeIngredientQuantities',
 'RecipeIngredientParts',
 'AggregatedRating',
 'ReviewCount',
 'Calories',
 'FatContent',
 'SaturatedFatContent',
 'CholesterolContent',
 'SodiumContent',
 'CarbohydrateContent',
 'FiberContent',
 'SugarContent',
 'ProteinContent',
 'RecipeServings',
 'RecipeYield',
 'RecipeInstructions']

In [20]:
recipe_column = ["RecipeId","AggregatedRating","ReviewCount"]

In [21]:
# Step 1: compute average rating per recipe
avg_rating = reviews.groupby('RecipeId')['Rating'].mean().reset_index()

# Step 2: rename column
avg_rating.rename(columns={'Rating': 'average_rating'}, inplace=True)

# Step 3: merge back
reviews_2 = reviews.merge(avg_rating, on='RecipeId', how='left')

reviews_2.head()

,ReviewId,RecipeId,AuthorId,AuthorName,Rating,Review,DateSubmitted,DateModified,source,average_rating
0,2,992,2008,gayg msft,5,better than any you can get at a restaurant!,2000-01-25T21:44:00Z,2000-01-25T21:44:00Z,foodcom,4.916667
1,7,4384,1634,Bill Hilbrich,4,"I cut back on the mayo, and made up the differ...",2001-10-17T16:49:59Z,2001-10-17T16:49:59Z,foodcom,4.500000
2,9,4523,2046,Gay Gilmore ckpt,2,i think i did something wrong because i could ...,2000-02-25T09:00:00Z,2000-02-25T09:00:00Z,foodcom,4.000000
3,13,7435,1773,Malarkey Test,5,easily the best i have ever had. juicy flavor...,2000-03-13T21:15:00Z,2000-03-13T21:15:00Z,foodcom,4.255814
4,14,44,2085,Tony Small,5,An excellent dish.,2000-03-28T12:51:00Z,2000-03-28T12:51:00Z,foodcom,4.545455


In [22]:
recipes = recipes[recipe_column]

In [23]:
reviews_2 = reviews.merge(recipes, on='RecipeId', how='left')

In [24]:
reviews_2.drop(columns=["DateSubmitted","AuthorName"], inplace=True)

In [25]:
reviews_2['DateModified'] = pd.to_datetime(
    reviews_2['DateModified'],
    format='mixed',
    errors='coerce'
)
reviews_2.head()

,ReviewId,RecipeId,AuthorId,Rating,Review,DateModified,source,AggregatedRating,ReviewCount
0,2,992,2008,5,better than any you can get at a restaurant!,2000-01-25 21:44:00+00:00,foodcom,5.0,15.0
1,7,4384,1634,4,"I cut back on the mayo, and made up the differ...",2001-10-17 16:49:59+00:00,foodcom,5.0,3.0
2,9,4523,2046,2,i think i did something wrong because i could ...,2000-02-25 09:00:00+00:00,foodcom,4.0,10.0
3,13,7435,1773,5,easily the best i have ever had. juicy flavor...,2000-03-13 21:15:00+00:00,foodcom,5.0,47.0
4,14,44,2085,5,An excellent dish.,2000-03-28 12:51:00+00:00,foodcom,5.0,23.0


In [33]:
df = comment_2.copy()

# 1. Select + rename columns
df_mapped = df.rename(columns={
    'recipe_id': 'RecipeId',
    'user_id': 'AuthorId',
    'rating': 'Rating',
    'comment': 'Review',
    'dateLastModified': 'DateModified',
    'aver_rate': 'AggregatedRating',
    'review_nums': 'ReviewCount'
})[[
    'RecipeId',
    'AuthorId',
    'Rating',
    'Review',
    'DateModified',
    'AggregatedRating',
    'ReviewCount',
    'source'
]]

# 2. Convert datetime (handle mixed formats)
df_mapped['DateModified'] = pd.to_datetime(
    df_mapped['DateModified'],
    format='mixed',
    errors='coerce',
    utc=True
)

# 3. Create ReviewId
df_mapped = df_mapped.reset_index(drop=True)
df_mapped['ReviewId'] = df_mapped.index

# 4. Reorder columns to match target format
df_mapped = df_mapped[[
    'ReviewId',
    'RecipeId',
    'AuthorId',
    'Rating',
    'Review',
    'DateModified',
    'source',
    'AggregatedRating',
    'ReviewCount'
]]

In [38]:
df_mapped.head()

,ReviewId,RecipeId,AuthorId,Rating,Review,DateModified,source,AggregatedRating,ReviewCount
0,0,222388,8542392,5,Best breakfast ever! I ran out of paprika whil...,2017-04-22 12:46:43.663000+00:00,foodrecsys,5.000000,3
1,1,222388,11174581,5,Awesome!\nIt's amazing.,2013-06-20 15:50:25.960000+00:00,foodrecsys,5.000000,3
2,2,222388,8262477,5,The flavors came together well and it really w...,2015-02-14 07:27:51.307000+00:00,foodrecsys,5.000000,3
3,3,240488,3574785,5,"Like most, I changed it a bit. Not a fan of T...",2017-10-07 18:20:08.973000+00:00,foodrecsys,4.764706,29
4,4,240488,12145410,2,This was one of the worst recipes I have ever ...,2018-01-06 00:06:09.563000+00:00,foodrecsys,4.764706,29


In [39]:
df_mapped.shape

(3794003, 9)

In [36]:
reviews_2.shape

(1401982, 9)

In [41]:
print(reviews_2.columns.tolist())
print(df_mapped.columns.tolist())

df_combined = pd.concat([reviews_2, df_mapped], ignore_index=True)

['ReviewId', 'RecipeId', 'AuthorId', 'Rating', 'Review', 'DateModified', 'source', 'AggregatedRating', 'ReviewCount']
['ReviewId', 'RecipeId', 'AuthorId', 'Rating', 'Review', 'DateModified', 'source', 'AggregatedRating', 'ReviewCount']


In [42]:
df_combined.shape

(5195985, 9)

In [46]:
def save_in_chunks_by_size(
    df,
    max_size_mb=100,
    output_dir="data/processed/Reviews",
    file_prefix="reviews_chunk",
):
    # convert to Path object
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    max_bytes = max_size_mb * 1024 * 1024

    current_chunk = []
    current_size = 0
    file_idx = 0

    for _, row in df.iterrows():
        row_df = pd.DataFrame([row])
        row_size = row_df.memory_usage(deep=True).sum()

        # if adding this row exceeds size → save current chunk
        if current_chunk and current_size + row_size > max_bytes:
            chunk_df = pd.concat(current_chunk, ignore_index=True)

            file_path = output_dir / f"{file_prefix}_{file_idx}.csv"
            chunk_df.to_csv(file_path, index=False)

            print(f"Saved: {file_path} ({round(current_size / 1e6, 2)} MB)")

            file_idx += 1
            current_chunk = []
            current_size = 0

        current_chunk.append(row_df)
        current_size += row_size

    # save last chunk
    if current_chunk:
        chunk_df = pd.concat(current_chunk, ignore_index=True)
        file_path = output_dir / f"{file_prefix}_{file_idx}.csv"
        chunk_df.to_csv(file_path, index=False)

        print(f"Saved: {file_path} ({round(current_size / 1e6, 2)} MB)")

In [48]:
from pathlib import Path
save_in_chunks_by_size(
    df_combined,
    max_size_mb=90,
    output_dir="../data/processed/Reviews",
    file_prefix="reviews_chunk"
)

Saved: ../data/processed/Reviews/reviews_chunk_0.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_1.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_2.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_3.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_4.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_5.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_6.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_7.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_8.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_9.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_10.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_11.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_12.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_13.csv (94.37 MB)
Saved: ../data/processed/Reviews/reviews_chunk_14.csv (94.37 MB)
Saved: ../data/processed/Reviews/re